# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process data defined by a [Croissant schema](https://mlcommons.org/croissant/) using the `mlcroissant` library. The workflow includes data loading, overview, extraction, processing, and visualization.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`:


In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load metadata and schema
dataset = mlc.Dataset(croissant_url)

print(f"Dataset title: {dataset.metadata.name}")
print("Description:")
print(dataset.metadata.description)


## 2. Data Overview
Explore available record sets and fields by enumerating their `@id` values. Each record set and field is uniquely identified by its `@id`.


In [ ]:
# List all available RecordSets and their structure
# Accessing dataset.metadata.record_sets (returns a dict {record_set_id: RecordSet})

record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} RecordSets.")
for record_set_id, record_set in record_sets.items():
    print(f"\nRecordSet @id: {record_set_id}")
    print(f"  Name: {getattr(record_set, 'name', None)}")
    print(f"  Description: {getattr(record_set, 'description', None)}")
    print(f"  Fields (@id):")
    for field_id, field in record_set.fields.items():
        print(f"    - {field_id} (name='{getattr(field, 'name', None)}', dataType={getattr(field, 'data_type', None)})")


## 3. Data Extraction
Choose record set(s) and load as DataFrames for further processing. Reference all entities using their `@id` (see above for available IDs).


In [ ]:
# Select the main record set for protein entries (adjust @id as discovered above)

# List all record set IDs, choose the best (often one with protein quantitative/identification)
record_set_ids = list(record_sets.keys())
print("All record set @id's:", record_set_ids)

# For this dataset, the quantitative data is usually in a RecordSet with name containing e.g. 'protein', 'quant', etc.
# You may need to adjust this variable based on the schema - set your main RecordSet @id below:
main_record_set_id = None
for rid, rs in record_sets.items():
    name = getattr(rs, 'name', '').lower() if getattr(rs, 'name', None) else ''
    if 'protein' in name or 'quant' in name:
        main_record_set_id = rid

# Fallback to the first RecordSet if not found by name
if main_record_set_id is None and record_set_ids:
    main_record_set_id = record_set_ids[0]

print(f"Using main record set: {main_record_set_id}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for {record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        if not df.empty:
            dataframes[record_set_id] = df
            print(f"  {df.shape[0]} rows, {df.shape[1]} columns")
        else:
            print("  No records or empty DataFrame.")
    except Exception as e:
        print(f"  Error: {e}")

# Show columns for the main record set
if main_record_set_id is not None and main_record_set_id in dataframes:
    print("\nColumns in main record set DataFrame (by @id):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print('No data extracted for the selected record set.')


## 4. Exploratory Data Analysis (EDA)

Select a numeric field (by `@id`) for analysis, filter, normalize, and group values. See above for possible column choices, such as peptide count, molecular weight, or normalized abundance fields.

In [ ]:
# Adjust these field @ids (columns) based on the main_record_set_id DataFrame above
df = dataframes[main_record_set_id]

# Find all numeric columns by dtype
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric field @ids:", numeric_cols)

# Choose a numeric field and group-by field by @id
if numeric_cols:
    numeric_field_id = numeric_cols[0]  # e.g. '@id:peptide_count' or similar
else:
    print("No numeric field found!")
    numeric_field_id = None

# Optionally, pick a field for grouping (e.g., protein name or some sample column)
possible_group_fields = [col for col in df.columns if col != numeric_field_id]
group_field = None
for col in possible_group_fields:
    if 'sample' in col.lower() or 'condition' in col.lower() or 'type' in col.lower():
        group_field = col
        break
if group_field is None and possible_group_fields:
    group_field = possible_group_fields[0]  # fallback

print(f"Using numeric field: {numeric_field_id}\nUsing group-by field: {group_field}")

if numeric_field_id:
    # Remove NAs for the numeric field
    filtered_df = df[df[numeric_field_id].notna()].copy()

    threshold = filtered_df[numeric_field_id].mean()
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records")

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Top 5 rows with {numeric_field_id} and normalized:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean_'+numeric_field_id)
        print(f"Mean {numeric_field_id} by {group_field} (top 5):")
        display(grouped_df.head())
else:
    print("No numeric field could be analyzed.")

## 5. Visualization

Visualize numeric field distributions and group means if available.

In [ ]:
# Histogram & boxplot for the main numeric field
if numeric_field_id:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    df[numeric_field_id].hist(bins=30)
    plt.title(f'Histogram: {numeric_field_id}')

    plt.subplot(1,2,2)
    df.boxplot(column=numeric_field_id)
    plt.title(f'Boxplot: {numeric_field_id}')
    plt.show()

    # If grouped, plot as bar
    if 'grouped_df' in locals():
        grouped_df.head(20).plot(kind='bar', legend=False)
        plt.title(f'Mean {numeric_field_id} by {group_field}')
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()
else:
    print('No numeric field for visualization.')

## 6. Conclusion

- We loaded and explored the protein quantification dataset from its Croissant metadata.
- Record set and field exploration was performed exclusively referenced by their `@id` fields.
- Numeric fields were analyzed and filtered, normalized, and visualized (histogram, boxplot, and barplots by group).
- This workflow demonstrates how `mlcroissant` enables transparent, structured dataset exploration directly from FAIR metadata schemas.